# Day 2 — Ridership Trends + Station Frequencies

**Estimated time:** ~3 hours  •  **Sections:** 3

## Learning objectives

- Load a multi-tab Excel workbook and reshape wide data into long
  format — a level up from the single-CSV morning session.
- Plot MARTA's ridership trend since 2019; identify the COVID dip
  and the fall-2023 faregate data-quality anomaly.
- Clean a messy station-level frequency file: fix inconsistent names,
  fill nulls, drop duplicates.
- Identify the two stadium-adjacent rail stations.

## Table of contents

1. [Section 1 — NTD level-up](#section-1) (~50 min)
2. [Section 2 — Ridership trends](#section-2) (~60 min)
3. [Section 3 — Station frequencies](#section-3) (~70 min)

---

## Setup

Run the next cell.


In [ ]:
# Setup — run this first. In Colab: clones the repo and installs deps.
# Locally: no-op (the conda env handles it).
import os, sys, urllib.request
try:
    import google.colab  # noqa: F401
    if not os.path.exists('/tmp/colab_bootstrap.py'):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/arctic-gsu/arctic-scisynth-2026-rwd/main/setup/colab_bootstrap.py',
            '/tmp/colab_bootstrap.py',
        )
    sys.path.insert(0, '/tmp')
    from colab_bootstrap import bootstrap
    bootstrap()
except ImportError:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
CHECKPOINTS = ROOT / 'data' / 'checkpoints'

# MARTA rail-adjacent constants — referenced throughout the week.
MARTA_NTD_ID = 40022  # 5-digit NTD ID. Older docs use '4022' — that is wrong.
MODE_NAMES = {'HR': 'Heavy Rail', 'MB': 'Bus', 'SR': 'Streetcar',
              'DR': 'Demand Response'}
STADIUM_ADJACENT = ['sec district', 'vine city']   # where fans alight
STADIUM_AREA = STADIUM_ADJACENT + ['five points']  # + transfer hub

print(f'✅ Imports ready. Working from {ROOT}')


<a id="section-1"></a>
## Section 1 — NTD level-up

In this morning's shared session you cleaned a single-tab CSV of MARTA
ridership. That CSV was extracted from a **multi-tab Excel workbook**
published monthly by the FTA National Transit Database. You're now
going to open that workbook directly and apply the same cleaning
skills at higher difficulty.

**MARTA's NTD ID is `40022`** (5-digit). You'll filter the giant
agency table down to just MARTA.


In [ ]:
NTD_SKIPROWS = 0  # Feb 2026 release has no preamble rows.
upt = pd.read_excel(RAW / 'ntd_monthly_ridership.xlsx',
                    sheet_name='UPT', skiprows=NTD_SKIPROWS)
print(f'UPT sheet shape: {upt.shape}')
print(f"First few columns: {list(upt.columns[:8])}")


### 🎯 Your Turn! Filter to MARTA only.


In [ ]:
# The setup cell defined MARTA_NTD_ID = 40022. Filter upt by that.
marta = upt[upt['NTD ID'] == ???].copy()
print(f'MARTA rows: {len(marta)}')
print(f"Modes MARTA reports: {list(marta['Mode'].unique())}")


<details><summary>💡 Click for solution</summary>

```python
marta = upt[upt['NTD ID'] == 40022].copy()
```

</details>


### Reshape to long format

The raw UPT sheet is **wide** — each month is a separate column. For
time-series plotting, we need **long** format: one row per
(mode, date) pair.


In [ ]:
meta_cols = ['NTD ID', 'Legacy NTD ID', 'Agency',
             'Mode/Type of Service Status', 'Reporter Type', 'UACE CD',
             'UZA Name', 'Mode', 'TOS', '3 Mode']
month_cols = [c for c in marta.columns if c not in meta_cols]

marta_long = marta.melt(
    id_vars=['Mode', 'TOS'],
    value_vars=month_cols,
    var_name='month_raw',
    value_name='upt',
)
marta_long['date'] = pd.to_datetime(marta_long['month_raw'], format='%m/%Y')
marta_long['upt'] = pd.to_numeric(marta_long['upt'], errors='coerce')
marta_long = marta_long.sort_values(['Mode', 'date']).reset_index(drop=True)

marta_long['mode_name'] = marta_long['Mode'].map(MODE_NAMES)
marta_long = marta_long.rename(columns={'Mode': 'mode',
                                        'TOS': 'type_of_service'})

print(f'Long format shape: {marta_long.shape}')
print(f"Date range: {marta_long['date'].min().date()} to {marta_long['date'].max().date()}")


### ✅ Checkpoint


In [ ]:
assert len(marta) > 0, '❌ MARTA filter returned zero rows. Check the NTD ID.'
assert 'HR' in marta_long['mode'].values, '❌ Expected a Heavy Rail (HR) row.'
assert marta_long['date'].dt.year.max() >= 2025, '❌ Data should go to 2025 or later.'
print(f"✅ {len(marta_long):,} rows, "
      f"{marta_long['mode'].nunique()} modes, through "
      f"{marta_long['date'].dt.year.max()}.")


### Save the cleaned ridership checkpoint


In [ ]:
marta_long_out = marta_long[['date', 'mode', 'mode_name',
                              'type_of_service', 'upt']]
marta_long_out.to_csv(CHECKPOINTS / 'day2_output_cleaned_ridership.csv',
                       index=False)
print(f'✅ Wrote {len(marta_long_out):,} rows to '
      'data/checkpoints/day2_output_cleaned_ridership.csv')


### 🔄 Recovery point

If your NTD cleanup went sideways, run the next cell to load a clean
pre-staged version and continue to Section 2.


In [ ]:
marta_long = pd.read_csv(CHECKPOINTS / 'day2_output_cleaned_ridership.csv',
                          parse_dates=['date'])
print(f'✅ Loaded {len(marta_long):,} rows. Ready for Section 2.')


---

<a id="section-2"></a>
## Section 2 — Ridership trends

Now plot ridership by mode from 2019 to present. You're looking for
three things:

1. The **COVID lockdown dip** in spring 2020.
2. MARTA's **ridership recovery** (or lack of it).
3. A **suspicious rail-ridership drop** starting around fall 2023.

MARTA has publicly acknowledged that the fall-2023 drop is **not a
real ridership decline** — it's a data-quality issue. Their Breeze-card
faregates stopped recording taps reliably. The NTD data reflects this
undercount.


In [ ]:
plot_data = marta_long[
    (marta_long['date'] >= '2019-01-01') &
    (marta_long['mode'].isin(['HR', 'MB', 'SR'])) &
    (marta_long['type_of_service'] == 'DO')
].copy()

fig, ax = plt.subplots(figsize=(14, 6))
for mode in ['HR', 'MB', 'SR']:
    d = plot_data[plot_data['mode'] == mode]
    ax.plot(d['date'], d['upt'], label=MODE_NAMES[mode], linewidth=1.5)

ax.axvline(pd.Timestamp('2020-03-15'), color='gray', ls='--', alpha=0.5,
           label='COVID lockdown')
ax.axvline(pd.Timestamp('2023-09-01'), color='red', ls='--', alpha=0.5,
           label='Faregate issues (approx)')
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Riders (UPT)')
ax.set_title('MARTA Monthly Ridership by Mode, 2019–Present')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Data-quality discussion

Look at the Heavy Rail (HR) line specifically. Starting around
September 2023, rail ridership drops sharply — but **bus ridership
stays stable**. If the drop were a real decline in MARTA usage, you'd
expect both modes to move together. That divergence is the fingerprint
of a faregate undercount, not a behavior change.

**As analysts, what do we do about it?**

- **Don't silently drop the bad months.** Flag the caveat in your
  briefing.
- **Benchmark against something else.** Super Bowl LIII (2019) gave
  MARTA a known-good "what the system can handle" ceiling: 155,000
  rail riders in one day.
- **Use ranges, not point estimates**, for any calculation that
  depends on current ridership.


### 🎯 Your Turn! Find the month with the lowest Heavy Rail UPT since 2019.


In [ ]:
# Filter plot_data to Heavy Rail, then find the row with the minimum upt.
# Hint: use .idxmin() on the upt column.
hr = plot_data[plot_data['mode'] == 'HR']
lowest = hr.loc[???]
print(f"Lowest HR month: {lowest['date'].date()} with {lowest['upt']:,.0f} UPT")


<details><summary>💡 Click for solution</summary>

```python
lowest = hr.loc[hr['upt'].idxmin()]
```

</details>


### ✅ Checkpoint


In [ ]:
assert lowest['date'].year == 2020, (
    f"❌ Expected the COVID lockdown (April 2020) to be the lowest month; "
    f"got {lowest['date'].date()}."
)
print(f'✅ COVID trough confirmed: {lowest["date"].date()}.')


### 🔄 Recovery point

Safe to run any time — reloads cleaned ridership from the checkpoint.


In [ ]:
marta_long = pd.read_csv(CHECKPOINTS / 'day2_output_cleaned_ridership.csv',
                          parse_dates=['date'])
print(f'✅ Loaded {len(marta_long):,} rows. Ready for Section 3.')


---

<a id="section-3"></a>
## Section 3 — Clean station frequencies, identify stadium stations

Now we turn to **`data/processed/station_frequencies.csv`** — the
messy per-station trains-per-hour file you previewed at the end of
Day 1. Your job: clean it, then identify the rail stations adjacent
to Mercedes-Benz Stadium.


In [ ]:
freq_raw = pd.read_csv(PROCESSED / 'station_frequencies.csv')
print(f'Raw shape: {freq_raw.shape}')
print(f"Nulls per column:\n{freq_raw.isna().sum()}")
print(f'\nDuplicated rows: {freq_raw.duplicated().sum()}')


### Cleaning recipe

1. **Normalize station names.** Some rows are uppercased. Some have a
   trailing `STATION`. Collapse multiple spaces, strip, title-case.
2. **Drop exact duplicate rows.**
3. **Fill null frequencies** with the median (a reasonable default
   for a small number of missing values).


In [ ]:
# 🎯 Your Turn! Fill in the three cleaning steps.
freq_clean = freq_raw.copy()

# 1. Normalize station names: strip whitespace, drop trailing ' STATION',
# collapse multiple spaces, title-case.
freq_clean['station_name'] = (
    freq_clean['station_name'].str.strip()
    .str.replace(r'\s+STATION$', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.???
)

# 2. Drop exact duplicate rows.
freq_clean = freq_clean.???

# 3. Fill null offpeak frequencies with the median.
median_op = freq_clean['offpeak_trains_per_hour'].median()
freq_clean['offpeak_trains_per_hour'] = (
    freq_clean['offpeak_trains_per_hour'].???(median_op)
)

print(f'Cleaned shape: {freq_clean.shape}')
freq_clean.head()


<details><summary>💡 Click for solution</summary>

```python
freq_clean['station_name'] = freq_clean['station_name'].str.title()
freq_clean = freq_clean.drop_duplicates()
freq_clean['offpeak_trains_per_hour'] = freq_clean['offpeak_trains_per_hour'].fillna(median_op)
```

</details>


### Identify stadium-adjacent stations

Mercedes-Benz Stadium sits southwest of downtown. The two closest rail
stations are **SEC District** (the former GWCC/CNN Center, ~0.32 km
away) and **Vine City** (~0.34 km). **Five Points** is the downtown
transfer hub where all four lines converge (~0.85 km from the
stadium); it's where fans transfer between lines, not a stadium-
adjacent alighting point.

For Day 4's demand model we'll focus on the **two adjacent stations**.


In [ ]:
# STADIUM_ADJACENT (2 stations) = fans alight here. This is what Day 4's
# capacity model uses, matching the Santanam et al. 2021 framing.
# STADIUM_AREA (3 stations) = adjacent + Five Points transfer hub, for display.
# Both constants are defined in the setup cell.
stadium_area = freq_clean[
    freq_clean['station_name'].str.lower().str.contains(
        '|'.join(STADIUM_AREA)
    )
].copy()

print('=== Stadium-area stations ===')
print(stadium_area.to_string(index=False))


### ✅ Checkpoint


In [ ]:
assert len(freq_clean) > 30, f'❌ Expected >30 stations; got {len(freq_clean)}.'
assert freq_clean['offpeak_trains_per_hour'].isna().sum() == 0, (
    '❌ Still have null offpeak values — check your fillna step.'
)
# Expect 3 matches: SEC District, Vine City, Five Points.
assert len(stadium_area) == 3, (
    f'❌ Expected 3 stadium-area matches; got {len(stadium_area)}. '
    'Check your keyword list.'
)
print(f'✅ {len(freq_clean)} clean rows, {len(stadium_area)} stadium-area stations.')


### Save today's checkpoint


In [ ]:
freq_clean.to_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv',
                   index=False)
print(f'✅ Wrote {len(freq_clean)} rows to '
      'data/checkpoints/day2_output_station_freq_clean.csv')


## What's next?

Tomorrow in **`day3_eda_and_mapping.ipynb`** you'll do a guided EDA
mini-lesson on the cleaned ridership data, then build an interactive
Folium map of MARTA with the World Cup venues overlaid.

See you then. 👋
